# DINO 물질 영역 분할 탐색 (multi-region)

이미지 *내부*의 물질 영역을 DINO patch feature 로 나눈다. 세 가지:
- **모드1a (label 없이, per-image):** patch feature → (PCA) → KMeans(K) 로 K개 영역 분할. 이미지마다 색 index 다름.
- **모드1b (전역):** 폴더 전체 patch 로 KMeans fit → 같은 물질=같은 색이 이미지 간 일치.
- **모드2 (prototype):** 참조 이미지+마스크로 물질별 대표 feature → cosine 최근접 분류.

정규화는 학습과 동일한 instance_norm(PerImageNormalize). 모델은 한 번만 로드하고 셀 반복 실험.

## 0. 설정 (여기만 수정)

In [ ]:
# ===== 경로/파라미터 =====
CONFIG_FILE = '../dino_v3/dinov3/configs/train/weaksup/stage2_ssl_weaksup_long.yaml'
WEIGHTS     = '/서버/teacher_checkpoint.pth'   # teacher(head 포함) 체크포인트
DATA_ROOT   = '/서버/이미지폴더'                # 분할할 이미지 폴더 (flat)
IMAGE_SIZE  = 448
N_BLOCKS    = 1        # 사용할 마지막 block 수
K           = 4        # 나눌 물질 영역 수 (multi-region)
N_PCA       = 10       # KMeans 전 PCA 축소 차원 (0=원 feature)
MAX_IMAGES  = 12
SEED        = 0

## 1. import + 헬퍼

In [ ]:
import os, sys, glob
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

# notebooks/ 에서 dino_v3 import 가능하게 경로 추가
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_DINOV3 = os.path.join(_ROOT, 'dino_v3')
for _p in (_DINOV3, _ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import dinov3.distributed as distributed
from dinov3.eval.setup import setup_and_build_model
from inspection.em_aug import build_em_eval_transform

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

@torch.inference_mode()
def extract_patch_map(model, x, n_blocks, autocast_dtype):
    '''x: [1,3,H,W] -> (patch feats [N,C], grid g).'''
    with torch.autocast('cuda', dtype=autocast_dtype):
        outs = model.get_intermediate_layers(x, n=n_blocks, reshape=False, return_class_token=True, norm=True)
    patch = outs[-1][0][0].float().cpu().numpy()   # [N,C] 마지막 block patch tokens
    n = patch.shape[0]; g = int(round(n ** 0.5))
    assert g * g == n, f'patch {n} 정사각 grid 아님 - image_size/patch_size 확인'
    return patch, g

def pca_rgb(feats, g):
    from sklearn.decomposition import PCA
    c = PCA(n_components=3).fit_transform(feats)
    c = (c - c.min(0)) / (np.ptp(c, axis=0) + 1e-8)
    return c.reshape(g, g, 3)

def upsample_nn(lbl, size):
    import cv2
    return cv2.resize(lbl.astype(np.int32), (size, size), interpolation=cv2.INTER_NEAREST)

def reduce_feats(x, n_pca, fit=None):
    from sklearn.decomposition import PCA
    if n_pca and 0 < n_pca < x.shape[1]:
        if fit is None:
            fit = PCA(n_components=n_pca).fit(x)
        return fit.transform(x), fit
    return x, fit

def show_row(orig, pca_img, seg_map, size, title):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(orig);      ax[0].set_title('input')
    ax[1].imshow(pca_img);   ax[1].set_title('PCA(3)-RGB')
    ax[2].imshow(upsample_nn(seg_map, size), cmap='tab20', interpolation='nearest')
    ax[2].set_title(title)
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()

print('헬퍼 준비 완료')

## 2. 모델 로드 (한 번만)

In [ ]:
if not distributed.is_enabled():
    distributed.enable(overwrite=True)

model, ctx = setup_and_build_model(
    config_file=CONFIG_FILE, pretrained_weights=WEIGHTS, output_dir='./_cluster_tmp')
model.cuda().eval()
autocast_dtype = ctx['autocast_dtype']
eval_tf = build_em_eval_transform(IMAGE_SIZE)
print('모델 로드 완료')

## 3. 이미지 로드 + feature 추출 (재사용)

In [ ]:
paths = sorted(p for p in glob.glob(os.path.join(DATA_ROOT, '*'))
               if p.lower().endswith(IMG_EXTS))[:MAX_IMAGES]
assert paths, f'이미지 없음: {DATA_ROOT}'

feats_list, grids, origs = [], [], []
for p in paths:
    img = Image.open(p).convert('RGB')
    x = eval_tf(img).unsqueeze(0).cuda(non_blocking=True)
    f, g = extract_patch_map(model, x, N_BLOCKS, autocast_dtype)
    feats_list.append(f); grids.append(g)
    origs.append(np.asarray(img.resize((IMAGE_SIZE, IMAGE_SIZE))))

print(f'{len(paths)}장, patch grid {grids[0]}x{grids[0]}, feat dim {feats_list[0].shape[1]}')

## 4a. 모드1a — per-image KMeans (label 없이, 이미지별 K영역)
PCA-RGB 와 나란히 보고 물질 영역이 갈리는지 확인. `K` 를 셀 위에서 바꿔 재실행.

In [ ]:
from sklearn.cluster import KMeans

for p, f, g, orig in zip(paths, feats_list, grids, origs):
    red, _ = reduce_feats(f, N_PCA)
    lbl = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(red).reshape(g, g)
    show_row(orig, pca_rgb(f, g), lbl, IMAGE_SIZE, f'KMeans k={K} (per-image)')

## 4b. 모드1b — 전역 KMeans (같은 물질=같은 색, 이미지 간 일치)
폴더 전체 patch 로 KMeans fit → cluster index 가 이미지 간 일관. 물질 라벨링에 더 유용.

In [ ]:
allf = np.concatenate(feats_list, 0)
redall, pca_fit = reduce_feats(allf, N_PCA)
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(redall)
print('전역 cluster 크기:', np.bincount(km.labels_, minlength=K).tolist())

for p, f, g, orig in zip(paths, feats_list, grids, origs):
    red, _ = reduce_feats(f, N_PCA, pca_fit)
    lbl = km.predict(red).reshape(g, g)
    show_row(orig, pca_rgb(f, g), lbl, IMAGE_SIZE, f'global KMeans k={K}')

## 4c. 적정 K 탐색 (silhouette)
물질 영역 수를 모르면 k별 silhouette 로 후보 확인 (높을수록 잘 갈림).

In [ ]:
from sklearn.metrics import silhouette_score

sample = allf if 'allf' in globals() else np.concatenate(feats_list, 0)
red, _ = reduce_feats(sample, N_PCA)
rng = np.random.default_rng(SEED)
sub = red[rng.choice(len(red), min(5000, len(red)), replace=False)]
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(sub)
    print(f'k={k}: silhouette={silhouette_score(sub, lab):.3f}')

## 5. 모드2 — prototype (참조 이미지+마스크로 물질별 대표 feature)
라벨된 참조 1장으로 물질별 prototype 을 만들고, 각 patch 를 cosine 최근접 물질로 분류.
마스크 pixel = 물질 class id. multi-class 그대로 동작.

In [ ]:
PROTO_REF  = '/서버/ref.png'
PROTO_MASK = '/서버/ref_mask.png'   # pixel = 물질 class id
PATCH_SIZE = 16
PURITY     = 0.8                     # 이 미만 순도 patch 는 prototype 에서 제외

from dinov3.data.labeled.patch_label import mask_to_patch_labels

rimg = Image.open(PROTO_REF).convert('RGB')
rx = eval_tf(rimg).unsqueeze(0).cuda()
rf, rg = extract_patch_map(model, rx, N_BLOCKS, autocast_dtype)

marr = np.array(Image.open(PROTO_MASK))
if marr.ndim == 3:
    marr = marr[..., 0]
# 마스크를 feature grid 에 맞춰 리사이즈 후 patch label
m_res = np.array(Image.fromarray(marr).resize((rg * PATCH_SIZE, rg * PATCH_SIZE), Image.NEAREST))
plabels = mask_to_patch_labels(torch.from_numpy(m_res.astype(np.int64)), PATCH_SIZE, PURITY, -1).numpy()

rf_n = rf / (np.linalg.norm(rf, axis=1, keepdims=True) + 1e-8)
classes = sorted(int(c) for c in np.unique(plabels) if c >= 0)
protos = np.stack([rf_n[plabels == c].mean(0) for c in classes])
protos /= (np.linalg.norm(protos, axis=1, keepdims=True) + 1e-8)
print('prototype classes:', classes, '| patches:', [int((plabels == c).sum()) for c in classes])

In [ ]:
for p, f, g, orig in zip(paths, feats_list, grids, origs):
    fn = f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)
    sim = fn @ protos.T                       # [N, n_class]
    pred = np.array([classes[i] for i in sim.argmax(1)]).reshape(g, g)
    show_row(orig, pca_rgb(f, g), pred, IMAGE_SIZE, 'prototype cosine (multi-class)')